In [1]:
import instructor
from enum import Enum
from openai import AzureOpenAI
from functools import partial
from response_model import PSOM_total,Neuro_Deficit_Score_Severity,Neuro_Deficit_Type,follow_up_status,post_discharge_rehabilitation, QuestionAnswer
import pandas as pd
from datetime import datetime
from tqdm import tqdm
import os
import re
from openpyxl import load_workbook
from openpyxl import Workbook
from typing import List, Optional, Literal, Type
from pydantic import BaseModel
from collections import Counter
from guidelines import GUIDELINES

In [2]:
# Define your Azure OpenAI configuration
model = "gpt4o_cursor"
engine = "gpt4o_cursor"
endpoint = "https://gpt4o-xj.openai.azure.com/"

api_version = "2024-02-15-preview"
api_type = "azure"

# key_name = "AZURE_KEY"
api_key = '15f19bd128a54ec7b2a10f46299272ec'

# Create the Azure OpenAI client
client = AzureOpenAI(
    api_version=api_version,
    api_key=api_key,
    azure_endpoint=endpoint,
    azure_deployment=engine
)

# Patch the client with instructor
client = instructor.patch(client)

# Optionally, create a partial function with default parameters
in_client = partial(
    client.chat.completions.create,
    model=model,
    temperature=0,
    max_retries=2  # Retries in case of API issues
)


In [3]:
output_directory = "output/"
note_directory = "notes/"
results_file_name = 'results.xlsx'
# sheetname = ['PSOM_total','Neuro_Deficit_Score_Severity','Neuro_Deficit_Type','follow_up_status','post_discharge_rehabilitation']
# entities = [PSOM_total,Neuro_Deficit_Score_Severity,Neuro_Deficit_Type,follow_up_status,post_discharge_rehabilitation]
# sheetname = ['Neuro_Deficit_Type','follow_up_status','post_discharge_rehabilitation']
# entities = [Neuro_Deficit_Type,follow_up_status,post_discharge_rehabilitation]
# sheetname = ['Neuro_Deficit_Score_Severity']
# entities = [Neuro_Deficit_Score_Severity]
# sheetname = ['post_discharge_rehabilitation']
# entities = [post_discharge_rehabilitation]
# sheetname = ['follow_up_status']
# entities = [follow_up_status]
# main_field = 'follow_up_status'
sheetname = ['Neuro_Deficit_Type']
entities = [Neuro_Deficit_Type]
main_field = 'neurologic_deficit_type'

In [4]:
# def generate_ask_ai(entity_model):

#     def ask_ai(notes: str):
#         try:
#             cleaned_notes = notes.strip()
#             prompt_template = f'''
# Extract information from the clinical notes.
# Here is the clinical notes: {cleaned_notes}
#             '''
#             # print(prompt_template)
#             # Making a call to the AI client with the appropriate parameters
#             response = in_client(
#                 response_model=entity_model,
#                 messages=[
#                     {
#                         "role": "system",
#                         "content": "You will help extract entities from clinical protocol documents and answer questions.",
#                     },
#                     {
#                         "role": "user",
#                         "content": prompt_template,
#                     },
#                 ],
#                 # validation_context={"text_chunk": notes},
#             )

#             return response

#         except Exception as e:
#             print(f"Error occurred: {e}")
#             return None
#     return ask_ai

In [5]:
# def ask_ai(question: str, notes: str) -> QuestionAnswer:
#     cleaned_notes = notes.strip()

#     response = in_client(
#         response_model = QuestionAnswer,
#         messages = [
#             {
#                 "role": "system",
#                 "content": "You are a clinical note reviewer to answer questions with correct and exact citations.",
#             },
#             {"role": "user", "content": f"{notes}"},
#             {"role": "user", "content": f"Question: {question}"},
#         ],
#         validation_context={"text_chunk": notes},
#     )
#     return response

# with open(note_directory + "30073-1.txt", 'r') as f:
#     notes = f.read()
# question = "After hospital discharge, did the patient receive any of rehabilitation?"
# response = ask_ai(question, notes)

In [6]:
guideline = GUIDELINES.get(Neuro_Deficit_Type.__name__)

In [7]:
guideline

'\n    Extract the neurologic deficit type from the clinical notes, strictly adhering to the following guidelines:\n\n    Do NOT infer neurologic deficit type from the patient\'s medical history. "History of" or "past medical history" should not be used to extract neurologic deficit type.\n\n    The presence of anxiety, depression, or other psychiatric disorders explicitly indicates a behavioral deficit type.\n\n    The phrases "chronic cognitive changes" or "not being able to explain work" or "decreased concentration" explicitly indicate a cognition deficit type.\n\n    The phrase "c/f delays" explicitly indicates a motor deficit type.\n\n    PSOM scores can explicitly indicate a neurologic deficit type. \n    If sensorimotor has a score greater than 0, it indicates a motor deficit. \n    If language production or comprehension has a score greater than 0, it indicates a speech deficit. \n    If comprehension behavioral has a score greater than 0, it indicates a cognition or behavior d

In [8]:
def ask_ai(notes: str, schema_class: Type[BaseModel]) -> BaseModel:
    # prompt = schema_class.__doc__
    SYSTEM_PROMPT = '''
    You are a clinical note reviewer. Given a clinical note and a question, follow these steps:

    1. Carefully read the clinical note. Think step by step.
    2. Identify any exact quotes (phrases) from the note that support a potential answer.
    3. From these quotes, infer a fact that answers the question.
    4. Based on the fact, answer the question.
    '''
    guideline = GUIDELINES.get(schema_class.__name__, "No specific guideline available for this response model.")

    cleaned_notes = notes.strip()
    response = in_client(
        response_model=schema_class,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": guideline},
            {"role": "user", "content": f"CLINICAL NOTE:\n{cleaned_notes}"},
            # {"role": "user", "content": f"TASK:\n{prompt.strip()}"},
        ],
        validation_context={"text_chunk": notes},
    )
    return response

In [9]:
def process_entity_value(value):
    """Process entity values, handling Enums and lists of Enums"""
    # Check if value is a non-empty list and contains Enums
    if isinstance(value, list) and len(value) > 0 and isinstance(value[0], Enum):
        # If the list is non-empty and contains Enums, extract the string representation
        return [v.value for v in value]
    elif isinstance(value, Enum):
        # If the value itself is an Enum, get its string representation
        return value.value
    return value

In [10]:
n_repeats = 1  

file_path = output_directory + results_file_name
if not os.path.exists(file_path):
    wb = Workbook()
    wb.save(file_path)
for i, Entity in enumerate(entities):
    results = []
    for filename in tqdm(os.listdir(note_directory)):
        print(filename)
        if filename.startswith('.'):
            continue
        with open(note_directory + filename, 'r') as f:
            notes = f.read()
        # ask_ai = generate_ask_ai(Entity)
        # single call
        if n_repeats == 1:
            try:
                response = ask_ai(notes, Entity)
                entity_dict = response.model_dump()
                entity_dict = {k: process_entity_value(v) for k, v in entity_dict.items()}
                newline = {'filename':filename}
                newline.update(entity_dict)

                results.append(newline)
            except Exception as e:
                print(f"Error occurred during API call: {e}")

        
        # multiple calls
        if n_repeats > 1:
            responses = []
            for _ in range(n_repeats):
                try:
                    response = ask_ai(notes, Entity)
                    responses.append(response)
                except Exception as e:
                    print(f"Error occurred during API call: {e}")
                    # Optionally, append a default or empty response to maintain consistency
                    responses.append(Entity())
            entity_dicts = [r.dict() for r in responses]

            # deal with Enums
            for ed in entity_dicts:
                for k, v in ed.items():
                    if isinstance(v, list):
                        ed[k] = [process_entity_value(i) for i in v]
                    else:
                        ed[k] = process_entity_value(v)
        
        
            # choose the most common answer and calculate consistency
            all_answers = [ed.get(main_field, None) for ed in entity_dicts]
            answer_counter = Counter(all_answers)

            most_common_answer, count = answer_counter.most_common(1)[0]
            consistency_score = round(count / n_repeats, 2)

            final_output = entity_dicts[0].copy()

            # choose the most common answer and output the first one
            for ed in entity_dicts:
                if ed.get(main_field, None) == most_common_answer:
                    final_output = ed.copy()
                    break
            newline = {'filename':filename}
            newline.update(final_output)
            newline.update({
                "answer_consistency": consistency_score,
                "all_answers": all_answers
            })
            results.append(newline)
    
    df = pd.DataFrame(results)  

    with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            # Write DataFrame to a new sheet
            df.to_excel(writer, sheet_name=sheetname[i])
    
# df.to_csv(output_directory + results_file_name, index=False)

  0%|          | 0/51 [00:00<?, ?it/s]

30078-1.txt


  2%|▏         | 1/51 [00:01<01:15,  1.52s/it]

30120-1.txt


  4%|▍         | 2/51 [00:02<01:03,  1.29s/it]

...

 96%|█████████▌| 49/51 [00:57<00:02,  1.41s/it]

30092-1.txt


 98%|█████████▊| 50/51 [00:58<00:01,  1.31s/it]

30089-1.txt


100%|██████████| 51/51 [00:59<00:00,  1.16s/it]


In [11]:
# # Save the DataFrame to an Excel file                 w33